In [2]:
import requests
import json
import altair as alt

In [39]:
### PREPARING OUR BASE SPEC

import json

# // Get out base spec (as above)
url = "https://raw.githubusercontent.com/ZhuolinLi-Jolin/ZhuolinLi-Jolin.github.io/refs/heads/main/archive/chart0.json"
# 注意：只请求一次即可
base_spec = requests.get(url).json()

# // 1. 保存原始的 mark 配置
original_mark = base_spec.get('mark', {})
if isinstance(original_mark, str):
    original_mark = {'type': original_mark}

# // 2. 设置标题
base_spec['title'] = {
    'text': "Inflation, consumer prices (annual %)",
    'fontSize': 11,
    'subtitleFontSize': 9
}

# // 3. 关键修改：添加数据过滤器 (Transform)
# 排除 2010 年，使用过滤器而不是硬编码坐标轴刻度
if 'transform' not in base_spec:
    base_spec['transform'] = []
base_spec['transform'].append({
    "filter": "year(datum.date) >= 2011"
})

# // 4. 配置编码 (Encoding)
# 确保 encoding 字典存在
if 'encoding' not in base_spec:
    base_spec['encoding'] = {}

# --- 修改 X 轴 ---
base_spec['encoding']['x'] = {
    'field': 'date',
    'type': 'temporal',  # 时间类型
    'title': '',
    'axis': {
        'labelFontSize': 8,
        'format': '%Y',     # 强制显示为年份格式
        'tickCount': 'year' # 自动按年显示刻度
    }
}

# --- 修改 Y 轴 ---
base_spec['encoding']['y'] = {
    'field': 'value',
    'type': 'quantitative',  # 定量类型
    'title': 'value',
    'axis': {
        'labelFontSize': 8,
        'titleFontSize': 9
    }
}

# --- 关键修改：修复鼠标悬停数据 (Tooltip) ---
# 显式定义 tooltip，这样鼠标放上去才会显示正确对应的 x 和 y 值
base_spec['encoding']['tooltip'] = [
    {'field': 'date', 'type': 'temporal', 'title': 'Date', 'format': '%Y'},
    {'field': 'value', 'type': 'quantitative', 'title': 'Value', 'format': '.2f'}
]

# // 5. 调整图表尺寸（等比例放大1.5倍）
base_spec['width'] = 286
base_spec['height'] = 198
base_spec['autosize'] = {'type': 'fit', 'contains': 'padding'}

# // 6. 调整折线图样式 (Mark)
base_spec['mark'] = {
    'type': 'line',
    'point': False,       # 不显示圆点
    'strokeWidth': 1.5,
    'color': original_mark.get('color', '#3d5a47')
}

# // 7. 全局配置 (Config)
base_spec['config'] = {
    'axis': {
        'labelFontSize': 8,
        'titleFontSize': 9
    },
    'legend': {
        'labelFontSize': 8,
        'titleFontSize': 9
    }
}

# // 8. 处理数据源
# 注意：删除 URL 后，你必须在后续步骤中向 base_spec['data']['values'] 注入实际数据
if 'data' in base_spec and 'url' in base_spec['data']:
    del base_spec['data']['url']

# // Print out our new Spec:
print(json.dumps(base_spec, indent=2))

{
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "width": 286,
  "height": 198,
  "autosize": {
    "type": "fit",
    "contains": "padding"
  },
  "background": "#fefdfb",
  "padding": 20,
  "title": {
    "text": "Inflation, consumer prices (annual %)",
    "fontSize": 11,
    "subtitleFontSize": 9
  },
  "data": {
    "values": [
      {
        "date": "2017-01-01",
        "value": 0.1
      },
      {
        "date": "2018-01-01",
        "value": 0.2
      },
      {
        "date": "2019-01-01",
        "value": 0.5
      },
      {
        "date": "2020-01-01",
        "value": 0.6
      }
    ]
  },
  "transform": [
    {
      "filter": "year(datum.date) >= 2011"
    },
    {
      "filter": "year(datum.date) >= 2011"
    }
  ],
  "mark": {
    "type": "line",
    "point": false,
    "strokeWidth": 1.5,
    "color": "#3d5a47"
  },
  "encoding": {
    "x": {
      "field": "date",
      "type": "temporal",
      "title": "",
      "axis": {
        "labelFo

In [40]:
# // Define our base url with the {} placeholder for the country code.
base_api = 'https://api.worldbank.org/v2/country/{}/indicator/FP.CPI.TOTL.ZG?format=json&date=2010:2024&per_page=160'

# // Create a list of countries we want to get data for:
countries = ['usa', 'can', 'ind', 'tcd','egy','gbr']

for idx, country in enumerate(countries, 1):
  ## Build the api that we want to use:
  apiToUse = base_api.format(country)
  
  ## Fetch data from API
  response = requests.get(apiToUse).json()
  
  ## Extract the actual data (second element in the response)
  data = response[1]
  
  ## Extract only date and value fields, and sort by date
  clean_data = [{'date': item['date'], 'value': item['value']} for item in data]
  clean_data = sorted(clean_data, key=lambda x: x['date'])
  
  ## Now build the chart spec with values instead of url:
  base_spec['data'] = {'values': clean_data}
  base_spec['title']['subtitle'] = country.upper()

  # /// Turn the spec into JSON
  specJSON = json.dumps(base_spec)

  # /// 先预览：Turn the json into an Altair chart and display it
  new_chart = alt.Chart.from_json(specJSON)
  new_chart.display()
  
  # /// 再存储：Save to file (当前目录已经是 portfolio 文件夹)
  with open(f'cc6-{idx}.json', 'w') as f:
      json.dump(base_spec, f, indent=2)
  
  print(f"✅ Saved: cc6-{idx}.json ({country.upper()})")


alt.Chart(...)

✅ Saved: cc6-1.json (USA)


alt.Chart(...)

✅ Saved: cc6-2.json (CAN)


alt.Chart(...)

...

✅ Saved: cc6-4.json (TCD)


alt.Chart(...)

✅ Saved: cc6-5.json (EGY)


alt.Chart(...)

✅ Saved: cc6-6.json (GBR)
